# 01 - Data audit

Refresh and inspect the historical international results and current 2026 World Cup fixture feeds. Set `FIFA_PREDICT_OFFLINE=1` to use populated local caches.

In [ ]:
import os
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from fifa_predict.pipeline import audit_sources

offline = os.getenv("FIFA_PREDICT_OFFLINE", "").lower() in {"1", "true", "yes"}
audit = audit_sources(refresh=not offline, offline=offline)
audit.summary()

In [ ]:
historical = audit.historical
world_cup = audit.world_cup

assert historical["date"].min() >= pd.Timestamp("2000-01-01", tz="UTC")
assert not historical[["date", "home_team", "away_team"]].isna().any().any()
assert not world_cup[["date", "home_team", "away_team", "source"]].isna().any().any()
world_cup[["date", "home_team", "away_team", "stage", "status", "source"]].head(12)

In [ ]:
result_counts = pd.Series({
    "Home win": (historical.home_score > historical.away_score).sum(),
    "Draw": (historical.home_score == historical.away_score).sum(),
    "Away win": (historical.home_score < historical.away_score).sum(),
})
sns.set_theme(style="whitegrid")
ax = result_counts.div(result_counts.sum()).plot.bar(color=["#3763a3", "#999999", "#c84b31"])
ax.set(title="Historical regulation-time result distribution", ylabel="Share", xlabel="")
plt.xticks(rotation=0)
plt.show()